# 01 — build

Builds every touched package. tsc exit codes and dist outputs are the
artifact under test — the never-merged PRs this branch replaces did
not even compile.


In [ ]:
import jaen_testkit as k
k.start_run('01-build')
print(k.CONFIG['repo_root'])

In [ ]:
PACKAGES = [
    ('jaen', 'packages/jaen', 'dist/index.d.ts'),
    ('gatsby-source-jaen', 'packages/gatsby-source-jaen', 'dist/gatsby-node.js'),
    ('gatsby-plugin-jaen', 'packages/gatsby-plugin-jaen', 'dist/gatsby/gatsby-node.js'),
    ('gatsby-jaen-emailwerk', 'packages/gatsby-jaen-emailwerk', None),
]

import os
with k.section('package builds'):
    for name, path, artifact in PACKAGES:
        with k.check('build %s' % name) as c:
            r = c.require(k.sh('yarn build', cwd=k.repo_path(path),
                               timeout=k.CONFIG['build_timeout'],
                               label='yarn build %s' % name))
            c.ok('rc=0 in %.0fs' % r.duration_s)
            if artifact:
                c.expect_true(os.path.isfile(k.repo_path(path, artifact)),
                              artifact)


In [ ]:
with k.section('typecheck'):
    with k.check('gatsby-plugin-jaen project typecheck (pages incl. accounts)') as c:
        r = k.sh('npx tsc --noEmit -p tsconfig.json',
                 cwd=k.repo_path('packages/gatsby-plugin-jaen'),
                 timeout=k.CONFIG['build_timeout'])
        # The repo carries a handful of known pre-existing errors outside the
        # touched files; fail only on errors in accounts/locales/zitadel areas.
        interesting = [line for line in r.stdout.splitlines()
                       if 'accounts' in line or 'locales' in line
                       or 'zitadel' in line or 'emailwerk' in line]
        c.expect_equal(len(interesting), 0,
                       'no type errors in the new surfaces')
        if interesting:
            c.detail('\n'.join(interesting[:20]))


In [ ]:
k.summary()
k.save_results('results-01-build.json')
rc = k.verdict()
assert rc == 0, 'run has FAILures — see the summary above'